In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
data_path = Path("UsedCarsSA_Unclean_EN.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Put {data_path.name} next to this notebook")

df = pd.read_csv(data_path)
df = df.loc[df["Price"] != "Negotiable"].copy()
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df = df.drop(columns=["Link", "Color", "Negotiable", "Condition"])
df = df.dropna(subset=["Origin", "Options", "Gear_Type"])
df["Engine_Size"] = df["Engine_Size"].fillna(
    df.groupby(["Make", "Type"])["Engine_Size"].transform("median")
)
df = df.dropna(subset=["Engine_Size", "Price"])
df = df.loc[(df["Price"] >= 5000) & (df["Mileage"] <= 1_000_000)].copy()

rare_category_maps = {}
for col in ["Make", "Type", "Region"]:
    counts = df[col].value_counts()
    retained = counts[counts >= 10].index.tolist()
    rare_category_maps[col] = [str(x) for x in retained]
    df[col] = df[col].where(df[col].isin(retained), "Other")

print("Rows after cleaning:", len(df))
display(df.head())

Rows after cleaning: 5432


,Make,Type,Year,Origin,Options,Engine_Size,Fuel_Type,Gear_Type,Mileage,Region,Price
0,Chrysler,C300,2018,Saudi,Full,5.7,Gas,Automatic,103000,Riyadh,114000
2,Nissan,Sunny,2019,Saudi,Standard,1.5,Gas,Automatic,72418,Riyadh,27500
3,Hyundai,Elantra,2019,Saudi,Standard,1.6,Gas,Automatic,114154,Riyadh,43000
4,Hyundai,Elantra,2019,Saudi,Semi Full,2.0,Gas,Automatic,41912,Riyadh,59500
5,Honda,Accord,2018,Saudi,Full,1.5,Gas,Automatic,39000,Riyadh,72000


In [4]:
X = df.drop(columns=["Price"]).copy()
y = df["Price"].copy()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_cols] = X[cat_cols].fillna("Unknown").astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", len(X_train), "Test:", len(X_test))
print("Features:", X.columns.tolist())

Train: 4345 Test: 1087
Features: ['Make', 'Type', 'Year', 'Origin', 'Options', 'Engine_Size', 'Fuel_Type', 'Gear_Type', 'Mileage', 'Region']


In [5]:
model = CatBoostRegressor(
    iterations=800, depth=6, learning_rate=0.05, l2_leaf_reg=1,
    random_seed=42, verbose=0, thread_count=2
)
model.fit(X_train, np.log1p(y_train), cat_features=cat_cols)
pred = np.maximum(np.expm1(model.predict(X_test)), 0)
print("Test MAE:", round(mean_absolute_error(y_test, pred), 2))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 2))
print("Test R2:", round(r2_score(y_test, pred), 4))

Test MAE: 13315.99
Test RMSE: 28612.83
Test R2: 0.8422


In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    cv_model = CatBoostRegressor(
        iterations=800, depth=6, learning_rate=0.05, l2_leaf_reg=1,
        random_seed=42, verbose=0, thread_count=2
    )
    cv_model.fit(X_tr, np.log1p(y_tr), cat_features=cat_cols)
    val_pred = np.maximum(np.expm1(cv_model.predict(X_val)), 0)
    cv_results.append({
        "Fold": fold,
        "MAE": mean_absolute_error(y_val, val_pred),
        "R2": r2_score(y_val, val_pred)
    })
cv_df = pd.DataFrame(cv_results)
display(cv_df.round(4))
print("Average CV MAE:", round(cv_df["MAE"].mean(), 2))
print("Average CV R2:", round(cv_df["R2"].mean(), 4))
print("R2 Standard Deviation:", round(cv_df["R2"].std(), 4))

,Fold,MAE,R2
0,1,11692.7422,0.8954
1,2,14040.2563,0.7594
2,3,13133.1697,0.8657
3,4,13421.5960,0.7875
4,5,13359.8935,0.8305


Average CV MAE: 13129.53
Average CV R2: 0.8277
R2 Standard Deviation: 0.0555


In [ ]:
model.save_model("car_price_model.cbm")
metadata = {
    "features": X.columns.tolist(),
    "categorical_features": cat_cols,
    "rare_category_maps": rare_category_maps,
    "categorical_values": {col: sorted(X[col].unique().tolist()) for col in cat_cols},
    "numeric_medians": {col: float(X[col].median()) for col in X.columns if col not in cat_cols},
    "target_transform": "log1p",
    "inverse_transform": "expm1",
}
with open("model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("Saved car_price_model.cbm and model_metadata.json")

Saved car_price_model.cbm and model_metadata.json
